# 02 - Postulate 1: State Space

**The state of an isolated quantum system is completely described by a state vector in a Hilbert space.**

In plain English: a quantum system is described by a vector. For a single qubit, that vector lives in a 2-dimensional complex vector space. For n qubits, it lives in a 2^n dimensional space.

We covered vectors in the linear algebra section. This postulate is saying: those vectors ARE the physics. The vector is the most complete description of the system that is physically possible.

In [ ]:
import numpy as np

## Hilbert Space

A Hilbert space is a vector space with an inner product. For quantum computing:

- **Single qubit**: 2D complex Hilbert space (C^2)
- **Two qubits**: 4D complex Hilbert space (C^4)
- **n qubits**: 2^n dimensional complex Hilbert space (C^{2^n})

The basis vectors are the computational basis states: |0>, |1> for one qubit, |00>, |01>, |10>, |11> for two qubits, and so on.

In [ ]:
# Single qubit Hilbert space: C^2
# Any qubit state is |psi> = alpha|0> + beta|1>
# where |alpha|^2 + |beta|^2 = 1

# Some valid qubit states
states = {
    "|0>":    np.array([1, 0], dtype=complex),
    "|1>":    np.array([0, 1], dtype=complex),
    "|+>":    np.array([1, 1], dtype=complex) / np.sqrt(2),
    "|->":    np.array([1, -1], dtype=complex) / np.sqrt(2),
    "|i>":    np.array([1, 1j], dtype=complex) / np.sqrt(2),
    "|-i>":   np.array([1, -1j], dtype=complex) / np.sqrt(2),
}

print("Valid qubit states (all normalized to 1):\n")
for name, state in states.items():
    norm = np.linalg.norm(state)
    print(f"{name:5s} = {state.round(4)}  |norm| = {norm:.4f}")

## Pure States vs Mixed States

A **pure state** is a state vector |psi>. It represents a system we have maximum knowledge about.

A **mixed state** is a statistical mixture of pure states. It represents uncertainty about which pure state the system is actually in. Mixed states are described by **density matrices**, not state vectors.

For now, we will work with pure states only. Density matrices come later with NISQ devices and noise.

In [ ]:
# Pure state: |psi> = (1/sqrt(2))|0> + (1/sqrt(2))|1>
psi = np.array([1/np.sqrt(2), 1/np.sqrt(2)], dtype=complex)

# Density matrix for a pure state: rho = |psi><psi|
rho_pure = np.outer(psi, psi.conj())
print("Pure state density matrix:")
print(rho_pure.round(4))

# Key property: for a pure state, Tr(rho^2) = 1
purity = np.trace(rho_pure @ rho_pure)
print(f"Purity Tr(rho^2) = {purity.real:.4f} (1 = pure)")

# Mixed state: 50% |0> and 50% |1> (classical coin flip, NOT superposition)
rho_mixed = 0.5 * np.outer([1,0], [1,0]) + 0.5 * np.outer([0,1], [0,1])
print("\nMixed state density matrix (50/50 classical mixture):")
print(rho_mixed.round(4))

purity_mixed = np.trace(rho_mixed @ rho_mixed)
print(f"Purity Tr(rho^2) = {purity_mixed.real:.4f} (less than 1 = mixed)")

print("\nNotice: |+> and the 50/50 mixture both give equal probabilities")
print("but they are DIFFERENT states. |+> has coherence (off-diagonal terms).")
print("The mixture does not. This matters for interference.")

## The Bloch Sphere

Any single qubit pure state can be written as:

`|psi> = cos(theta/2)|0> + e^(i*phi) * sin(theta/2)|1>`

Where theta (0 to pi) and phi (0 to 2*pi) define a point on a sphere called the Bloch sphere.

- North pole (theta=0): |0>
- South pole (theta=pi): |1>
- Equator (theta=pi/2): superposition states like |+>, |->, |i>, |-i>

Every single qubit gate is a rotation on this sphere.

In [ ]:
def state_from_bloch(theta, phi):
    """Create a qubit state from Bloch sphere angles."""
    return np.array([
        np.cos(theta / 2),
        np.exp(1j * phi) * np.sin(theta / 2)
    ], dtype=complex)

def bloch_coords(state):
    """Get x, y, z coordinates on Bloch sphere from a state vector."""
    # Pauli expectation values
    X = np.array([[0, 1], [1, 0]], dtype=complex)
    Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    Z = np.array([[1, 0], [0, -1]], dtype=complex)
    
    x = np.real(state.conj() @ X @ state)
    y = np.real(state.conj() @ Y @ state)
    z = np.real(state.conj() @ Z @ state)
    return x, y, z

# Test: known states and their Bloch sphere positions
test_states = {
    "|0>  (north pole)": state_from_bloch(0, 0),
    "|1>  (south pole)": state_from_bloch(np.pi, 0),
    "|+>  (+x axis)":    state_from_bloch(np.pi/2, 0),
    "|->  (-x axis)":    state_from_bloch(np.pi/2, np.pi),
    "|i>  (+y axis)":    state_from_bloch(np.pi/2, np.pi/2),
    "|-i> (-y axis)":    state_from_bloch(np.pi/2, 3*np.pi/2),
}

print(f"{'State':<20s} {'x':>6s} {'y':>6s} {'z':>6s}")
print("-" * 42)
for name, state in test_states.items():
    x, y, z = bloch_coords(state)
    print(f"{name:<20s} {x:>6.2f} {y:>6.2f} {z:>6.2f}")

## Global Phase Does Not Matter

If you multiply an entire state by a complex number with modulus 1 (a global phase), the physics does not change. The probabilities stay the same.

`|psi>` and `e^(i*theta) |psi>` represent the same physical state.

But a RELATIVE phase between components (like |+> vs |->) does matter. That is what makes interference work.

In [ ]:
psi = np.array([1/np.sqrt(2), 1/np.sqrt(2)], dtype=complex)

# Multiply by global phase e^(i*pi/4)
global_phase = np.exp(1j * np.pi/4)
psi_phased = global_phase * psi

print("Original state:      ", psi.round(4))
print("With global phase:   ", psi_phased.round(4))
print()

# Same probabilities
print("Probabilities (original):")
print(f"  P(0) = {abs(psi[0])**2:.4f}, P(1) = {abs(psi[1])**2:.4f}")
print("Probabilities (with global phase):")
print(f"  P(0) = {abs(psi_phased[0])**2:.4f}, P(1) = {abs(psi_phased[1])**2:.4f}")
print("\nSame probabilities. Global phase is physically meaningless.")

# But RELATIVE phase matters
ket_plus = np.array([1, 1], dtype=complex) / np.sqrt(2)   # relative phase = 0
ket_minus = np.array([1, -1], dtype=complex) / np.sqrt(2)  # relative phase = pi

print("\n|+> and |-> have different RELATIVE phases:")
print(f"  |+> Bloch coords: {bloch_coords(ket_plus)}")
print(f"  |-> Bloch coords: {bloch_coords(ket_minus)}")
print("Different states, different physics.")

## Exercises

1. Create a qubit state on the Bloch sphere at theta=pi/3, phi=pi/4. What are the probabilities of measuring |0> and |1>?

2. Compute the density matrix of the state |->. Verify it is a pure state (purity = 1).

3. Convince yourself that |+> and the 50/50 mixed state are different by applying the Hadamard gate to both density matrices and comparing the output.

In [ ]:
# Your code here
